<a href="https://colab.research.google.com/github/sonjoy1s/CNN_learn/blob/main/Rice_Image_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
!pip install torch
!pip install torchvision

In [86]:
import os
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import ImageFolder
from PIL import Image
import torch.nn as nn
import torch.optim as optim
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [87]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [88]:
dataset_path =kagglehub.dataset_download("muratkokludataset/rice-image-dataset")
path = os.path.join(dataset_path, "Rice_Image_Dataset")
images = []
labels = []

for cls in os.listdir(path):
    cls_path = os.path.join(path, cls)
    if not os.path.isdir(cls_path):
        continue

    for img_name in os.listdir(cls_path):
        images.append(os.path.join(cls_path, img_name))
        labels.append(cls)

df = pd.DataFrame({
    'image': images,
    'label': labels
})

Using Colab cache for faster access to the 'rice-image-dataset' dataset.


In [89]:
df['label_id'], uniques = pd.factorize(df['label'])
print("Class Names:", uniques.tolist())

Class Names: ['Karacadag', 'Basmati', 'Jasmine', 'Arborio', 'Ipsala']


In [90]:
X_train,X_test= train_test_split(df['label_id'],test_size=0.2,random_state=42)

In [91]:
print(len(X_train))

60000


In [92]:
len(X_test)

15000

In [93]:
X_train.head()

,label_id
1200,0
64130,4
11518,0
70523,4
34803,2


In [94]:
transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5)*3,std=(0.5)*3)
    ]
)

In [95]:
from genericpath import isfile

class MyCustomeData(Dataset):
    def __init__(self, dataframe, transform=None):
        self.paths = dataframe['image'].values
        self.labels = dataframe['label_id'].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        img_path = self.paths[idx]
        img = Image.open(img_path).convert("RGB")

        label = int(self.labels[idx])

        if self.transform:
            img = self.transform(img)

        return img, label



In [96]:
train_dataset_full = MyCustomeData(df.loc[X_train.index], transform)
test_dataset_full = MyCustomeData(df.loc[X_test.index], transform)
num_classes = len(uniques)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 5
Full train size: 60000
Full test size: 15000


In [97]:
train_dataset = Subset(train_dataset_full, list(range (100, len(train_dataset_full))))
test_dataset = Subset(test_dataset_full, list(range(100, len(test_dataset_full))))

In [98]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

In [99]:
class MyCNNs(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [100]:
model = MyCNNs(num_classes=num_classes).to(device)

In [102]:
learning_rate = 0.0001
epochs = 5
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 0.6607
